In [ ]:
%%writefile logic.py
import itertools


class Sentence:

    def evaluate(self, model):
        raise Exception("nothing to evaluate")

    def formula(self):
        return ""

    def symbols(self):
        return set()

    @classmethod
    def validate(cls, sentence):
        if not isinstance(sentence, Sentence):
            raise TypeError("must be a logical sentence")

    @classmethod
    def parenthesize(cls, s):
        def balanced(s):
            count = 0
            for c in s:
                if c == "(":
                    count += 1
                elif c == ")":
                    if count <= 0:
                        return False
                    count -= 1
            return count == 0

        if not len(s) or s.isalpha() or (
            s[0] == "(" and s[-1] == ")" and balanced(s[1:-1])
        ):
            return s
        else:
            return f"({s})"


class Symbol(Sentence):

    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        return isinstance(other, Symbol) and self.name == other.name

    def __hash__(self):
        return hash(("symbol", self.name))

    def __repr__(self):
        return self.name

    def evaluate(self, model):
        try:
            return bool(model[self.name])
        except KeyError:
            raise Exception(f"variable {self.name} not in model")

    def formula(self):
        return self.name

    def symbols(self):
        return {self.name}


class Not(Sentence):
    def __init__(self, operand):
        Sentence.validate(operand)
        self.operand = operand

    def __eq__(self, other):
        return isinstance(other, Not) and self.operand == other.operand

    def __hash__(self):
        return hash(("not", hash(self.operand)))

    def __repr__(self):
        return f"Not({self.operand})"

    def evaluate(self, model):
        return not self.operand.evaluate(model)

    def formula(self):
        return "¬" + Sentence.parenthesize(self.operand.formula())

    def symbols(self):
        return self.operand.symbols()


class And(Sentence):
    def __init__(self, *conjuncts):
        for conjunct in conjuncts:
            Sentence.validate(conjunct)
        self.conjuncts = list(conjuncts)

    def add(self, conjunct):
        Sentence.validate(conjunct)
        self.conjuncts.append(conjunct)

    def __eq__(self, other):
        return isinstance(other, And) and self.conjuncts == other.conjuncts

    def __hash__(self):
        return hash(
            ("and", tuple(hash(conjunct) for conjunct in self.conjuncts))
        )

    def __repr__(self):
        conjunctions = ", ".join([str(conjunct) for conjunct in self.conjuncts])
        return f"And({conjunctions})"

    def evaluate(self, model):
        return all(conjunct.evaluate(model) for conjunct in self.conjuncts)

    def formula(self):
        if len(self.conjuncts) == 1:
            return self.conjuncts[0].formula()
        return " ∧ ".join(
            [Sentence.parenthesize(conjunct.formula()) for conjunct in self.conjuncts]
        )

    def symbols(self):
        return set.union(*[conjunct.symbols() for conjunct in self.conjuncts]) if self.conjuncts else set()


class Or(Sentence):
    def __init__(self, *disjuncts):
        for disjunct in disjuncts:
            Sentence.validate(disjunct)
        self.disjuncts = list(disjuncts)

    def __eq__(self, other):
        return isinstance(other, Or) and self.disjuncts == other.disjuncts

    def __hash__(self):
        return hash(
            ("or", tuple(hash(disjunct) for disjunct in self.disjuncts))
        )

    def __repr__(self):
        disjuncts = ", ".join([str(disjunct) for disjunct in self.disjuncts])
        return f"Or({disjuncts})"

    def evaluate(self, model):
        return any(disjunct.evaluate(model) for disjunct in self.disjuncts)

    def formula(self):
        if len(self.disjuncts) == 1:
            return self.disjuncts[0].formula()
        return " ∨ ".join(
            [Sentence.parenthesize(disjunct.formula()) for disjunct in self.disjuncts]
        )

    def symbols(self):
        return set.union(*[disjunct.symbols() for disjunct in self.disjuncts]) if self.disjuncts else set()


class Implication(Sentence):
    def __init__(self, antecedent, consequent):
        Sentence.validate(antecedent)
        Sentence.validate(consequent)
        self.antecedent = antecedent
        self.consequent = consequent

    def __eq__(self, other):
        return (
            isinstance(other, Implication)
            and self.antecedent == other.antecedent
            and self.consequent == other.consequent
        )

    def __hash__(self):
        return hash(("implies", hash(self.antecedent), hash(self.consequent)))

    def __repr__(self):
        return f"Implication({self.antecedent}, {self.consequent})"

    def evaluate(self, model):
        return ((not self.antecedent.evaluate(model))
                or self.consequent.evaluate(model))

    def formula(self):
        antecedent = Sentence.parenthesize(self.antecedent.formula())
        consequent = Sentence.parenthesize(self.consequent.formula())
        return f"{antecedent} => {consequent}"

    def symbols(self):
        return set.union(self.antecedent.symbols(), self.consequent.symbols())


class Biconditional(Sentence):
    def __init__(self, left, right):
        Sentence.validate(left)
        Sentence.validate(right)
        self.left = left
        self.right = right

    def __eq__(self, other):
        return (
            isinstance(other, Biconditional)
            and self.left == other.left
            and self.right == other.right
        )

    def __hash__(self):
        return hash(("biconditional", hash(self.left), hash(self.right)))

    def __repr__(self):
        return f"Biconditional({self.left}, {self.right})"

    def evaluate(self, model):
        return ((self.left.evaluate(model) and self.right.evaluate(model))
                or (not self.left.evaluate(model) and not self.right.evaluate(model)))

    def formula(self):
        left = Sentence.parenthesize(str(self.left))
        right = Sentence.parenthesize(str(self.right))
        return f"{left} <=> {right}"

    def symbols(self):
        return set.union(self.left.symbols(), self.right.symbols())


def model_check(knowledge, query):
    def check_all(knowledge, query, symbols, model):
        if not symbols:
            if knowledge.evaluate(model):
                return query.evaluate(model)
            return True

        remaining = symbols.copy()
        p = remaining.pop()

        model_true = model.copy()
        model_true[p] = True

        model_false = model.copy()
        model_false[p] = False

        return (check_all(knowledge, query, remaining, model_true) and
                check_all(knowledge, query, remaining, model_false))

    symbols = set.union(knowledge.symbols(), query.symbols())
    return check_all(knowledge, query, symbols, dict())

Writing logic.py


In [ ]:
%%writefile puzzle.py
from logic import *

# Símbolos
AKnight = Symbol("A is a Knight")
AKnave = Symbol("A is a Knave")

BKnight = Symbol("B is a Knight")
BKnave = Symbol("B is a Knave")

CKnight = Symbol("C is a Knight")
CKnave = Symbol("C is a Knave")


def character_rules(knight, knave):
    """
    Cada personaje es o caballero o mentiroso, pero no ambos.
    """
    return And(
        Or(knight, knave),
        Not(And(knight, knave))
    )


def says(knight_symbol, knave_symbol, statement):
    """
    Si el personaje es knight, lo que dice es verdadero.
    Si el personaje es knave, lo que dice es falso.
    """
    return And(
        Implication(knight_symbol, statement),
        Implication(knave_symbol, Not(statement))
    )


# ==========================================================
# Puzzle 0
# A says: "I am both a knight and a knave."
# ==========================================================
statement0 = And(AKnight, AKnave)

knowledge0 = And(
    character_rules(AKnight, AKnave),
    says(AKnight, AKnave, statement0)
)


# ==========================================================
# Puzzle 1
# A says: "We are both knaves."
# B says nothing.
# ==========================================================
statement1A = And(AKnave, BKnave)

knowledge1 = And(
    character_rules(AKnight, AKnave),
    character_rules(BKnight, BKnave),
    says(AKnight, AKnave, statement1A)
)


# ==========================================================
# Puzzle 2
# A says: "We are the same kind."
# B says: "We are of different kinds."
# ==========================================================
statement2A = Or(
    And(AKnight, BKnight),
    And(AKnave, BKnave)
)

statement2B = Or(
    And(AKnight, BKnave),
    And(AKnave, BKnight)
)

knowledge2 = And(
    character_rules(AKnight, AKnave),
    character_rules(BKnight, BKnave),
    says(AKnight, AKnave, statement2A),
    says(BKnight, BKnave, statement2B)
)


# ==========================================================
# Puzzle 3
# A says either "I am a knight." or "I am a knave.", but you don't know which.
# B says "A said 'I am a knave'."
# B then says "C is a knave."
# C says "A is a knight."
#
# Modelamos las dos posibilidades de lo que A pudo haber dicho:
#   - A dijo "I am a knight"   -> statement3A1 = AKnight
#   - A dijo "I am a knave"    -> statement3A2 = AKnave
#
# Como no sabemos cuál dijo, una de esas dos ocurrió.
# ==========================================================

statement3A1 = AKnight
statement3A2 = AKnave

# "A dijo 'soy knave'"
ASaidKnave = Symbol("A said 'I am a knave'")
ASaidKnight = Symbol("A said 'I am a knight'")

statement3B1 = ASaidKnave
statement3B2 = CKnave
statement3C = AKnight

knowledge3 = And(
    character_rules(AKnight, AKnave),
    character_rules(BKnight, BKnave),
    character_rules(CKnight, CKnave),

    # A dijo exactamente una de las dos frases
    Or(ASaidKnight, ASaidKnave),
    Not(And(ASaidKnight, ASaidKnave)),

    # Si A es knight, su frase debe ser verdadera; si es knave, falsa
    Implication(ASaidKnight, says(AKnight, AKnave, statement3A1)),
    Implication(ASaidKnave, says(AKnight, AKnave, statement3A2)),

    # Lo que B dice
    says(BKnight, BKnave, statement3B1),
    says(BKnight, BKnave, statement3B2),

    # Lo que C dice
    says(CKnight, CKnave, statement3C)
)


def main():
    symbols = [AKnight, AKnave, BKnight, BKnave, CKnight, CKnave]
    puzzles = [
        ("Puzzle 0", knowledge0),
        ("Puzzle 1", knowledge1),
        ("Puzzle 2", knowledge2),
        ("Puzzle 3", knowledge3)
    ]

    for puzzle, knowledge in puzzles:
        print(puzzle)
        for symbol in symbols:
            if model_check(knowledge, symbol):
                print(f"    {symbol}")
        print()


if __name__ == "__main__":
    main()

Writing puzzle.py


In [ ]:
!python puzzle.py

Puzzle 0
    A is a Knave

Puzzle 1
    A is a Knave
    B is a Knight

Puzzle 2
    A is a Knave
    B is a Knight

Puzzle 3
    A is a Knight
    B is a Knave
    C is a Knight

